# BNNPrior showcase (M1)

Demonstrates `anytimeacquisition.priors.bnn.BNNPrior`: the vectorized, ECDF-normalized BNN prior/environment, including everything added on 2026-08-27/28 to align it with PFNs4BO's and ifBO's own BNN priors.

Background, in order:
- `docs/log/2026-08-27-bnn-prior-flat-draws-crit-scaling.md` — why most draws were flat, and the `crit` fix
- `docs/log/2026-08-27-pfns4bo-bnn-prior-comparison.md` (+ 2026-08-28 addendum) — comparison against PFNs4BO's/ifBO's own BNN priors
- `docs/log/2026-08-28-align-bnn-prior-with-pfns4bo-ifbo.md` — noise, input scaling, sparseness, spurious dims, deeper `depth_range`, and the `log_amp_range` fix this surfaced
- `docs/milestones/M1.md` — checklist this notebook is showcasing

In [ ]:
import torch

from anytimeacquisition.priors.bnn import BNNPrior, plot_1d_environments, plot_2d_environments

torch.manual_seed(0)

## Basic construction and evaluation

`batch_size` independent environments (architectures) at once; `evaluate(x)` is ECDF-normalized to `[0, 1]` and differentiable w.r.t. `x`.

In [ ]:
prior = BNNPrior(batch_size=4, x_dim=3, seed=0, cache_dir=None)

x = torch.rand(4, 100, 3)
y = prior.evaluate(x)
print("depths:", prior.depth.tolist())
print("widths:", prior.width.tolist())
print("evaluate() output range:", y.min().item(), y.max().item())

## Differentiability (needed by M5's exploit/explore search)

In [ ]:
x_grad = torch.rand(4, 8, 3, requires_grad=True)
y_grad = prior.evaluate(x_grad, noise=False)
y_grad.sum().backward()
print("d evaluate / dx nonzero entries:", (x_grad.grad.abs() > 0).sum().item(), "/", x_grad.grad.numel())

## Noise toggle

`noise=True` (default): fresh preactivation + output noise each call. `noise=False`: deterministic at the current weights — what M5's search will want, so it differentiates through an exact, reproducible surface rather than a fresh noise draw every step.

In [ ]:
y_det_1 = prior.evaluate(x, noise=False)
y_det_2 = prior.evaluate(x, noise=False)
print("noise=False, identical across calls:", torch.equal(y_det_1, y_det_2))

y_noisy_1 = prior.evaluate(x, noise=True)
y_noisy_2 = prior.evaluate(x, noise=True)
print("noise=True, identical across calls:", torch.equal(y_noisy_1, y_noisy_2))

## Sparseness

`sparseness=0.145` (PFNs4BO/ifBO's own value): each hidden-to-hidden weight is zeroed with this probability, survivors rescaled by `1/sqrt(1-sparseness)`.

In [ ]:
sparse = BNNPrior(batch_size=4, x_dim=3, seed=1, sparseness=0.145, cache_dir=None)
dense = BNNPrior(batch_size=4, x_dim=3, seed=1, sparseness=0.0, cache_dir=None)
print(f"fraction of W_h exactly zero, sparseness=0.145: {(sparse.W_h == 0).float().mean().item():.3f}")
print(f"fraction of W_h exactly zero, sparseness=0.0:   {(dense.W_h == 0).float().mean().item():.3f}")

## Spurious (irrelevant) input dimensions

PFNs4BO §5.2: each input dim is independently "relevant" with probability `frac_relevant_features` (0.7 here, matching their 30% irrelevant); irrelevant dims' `W_in` row is zeroed per instance — literally not fed to the network, not just given a dummy value. Gradient w.r.t. an irrelevant dim should be exactly zero.

In [ ]:
spurious_prior = BNNPrior(batch_size=16, x_dim=4, seed=2, frac_relevant_features=0.5, cache_dir=None)
x_s = torch.rand(16, 10, 4, requires_grad=True)
y_s = spurious_prior.evaluate(x_s, noise=False)
y_s.sum().backward()

irrelevant = spurious_prior.relevant_mask == 0
grad_per_dim = x_s.grad.abs().sum(dim=1)  # [B, d]
print("relevant_mask (1=relevant) for first 4 instances:\n", spurious_prior.relevant_mask[:4])
print("gradient at irrelevant dims is exactly zero:", (grad_per_dim[irrelevant] == 0).all().item())
print("gradient at relevant dims is nonzero:", (grad_per_dim[~irrelevant] > 0).all().item())

## 1D environments

True curve on a dense grid + random ECDF-normalized query points, for a handful of independently drawn environments.

In [ ]:
_ = plot_1d_environments(n_envs=4, n_random=14, seed=9)

## 2D environments

Same idea as a heatmap. Note the `log_amp_range` fix (2026-08-28): before it, raising `depth_range`'s ceiling produced visibly pure-noise environments here at high depth — caught by looking at this exact plot, not by any aggregate metric.

In [ ]:
_ = plot_2d_environments(n_envs=4, n_random=30, grid_res=80, seed=9)

## Episode sampling (for M2's PFN training)

In [ ]:
x_tr, y_tr, x_te, y_te = prior.sample_episode(n_train=10, n_test=5)
print("train:", x_tr.shape, y_tr.shape)
print("test: ", x_te.shape, y_te.shape)